# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">5주차 · 전부 계산할 수는 없다 — 먼저 추리고, 그다음 정렬하기</mark>

지난주에는 한 사람에게 추천을 만들려고 **종목 100개를 전부** 점수 매겼습니다.

오늘은 순서를 바꿉니다. **먼저 싼 방법으로 40개만 추리고, 그 40개만 지난주 모델로 정렬합니다.**

**오늘 모델은 한 글자도 고치지 않습니다.** 바뀌는 것은 **그 계산을 몇 개에 하느냐** 하나뿐입니다.

---

### 오늘의 구성

| 파트 | 종류 | 어디서 | 하는 일 |
|---|---|---|---|
| 1 | 개념 | 슬라이드 | 2단계 추천 · 후보 Recall 이라는 천장 · Two-Tower · ANN 을 이해한다 |
| 2 | 실습 | **이 노트북 5.1~5.8** | 후보를 뽑고, 후보에만 점수를 매기고, 후보 개수를 바꿔 본다 |
| 3 | 마무리 | 슬라이드 | 오늘의 용어 · 점수판 · 다음 주 |

> **슬라이드에서 원리를 이해하고, 노트북에서는 실제로 작동하는지 확인합니다.** 채점은 3주차와 같은 `recsys.recall_at_k`를 그대로 사용합니다.

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기 · 이 노트북을 여는 법</mark>

이 노트북은 **아무것도 설치하지 않고** 브라우저에서 바로 실행할 수 있습니다.

1. https://colab.research.google.com/github/welovecherry/recsys/blob/main/notebooks/05_two_stage.ipynb
2. 구글 계정으로 로그인합니다.
3. **경고창이 뜨면 `Run anyway` 를 누릅니다.**
4. **아래 "실습 준비" 셀의 ▶ 버튼을 누릅니다.** 실습 자료를 받아옵니다. 10초쯤 걸립니다.
5. 그다음부터는 위에서 아래로 셀을 하나씩 실행하면 됩니다.

> ⚠ **고친 내용을 남기려면** 메뉴에서 `파일 → 드라이브에 사본 저장` 을 눌러 주세요.

---

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">먼저 이 셀부터 실행하세요  ▶</mark>

In [1]:
# 이 셀에서 하는 일 — Colab 에서 열었으면 실습 자료를 내려받는다
# 왜 하나 — Colab 은 열 때마다 빈 컴퓨터라 데이터·recsys.py 가 없다. 내 컴퓨터면 그냥 넘어간다

import os          # 폴더를 만들고 옮겨 다니는 도구
import sys         # 지금 파이썬이 어떤 환경인지 알려 주는 도구
import subprocess  # 터미널 명령을 파이썬에서 대신 실행해 주는 도구

if "google.colab" in sys.modules:                    # Colab 이면 이 안이 실행된다
    if os.path.exists("/content/recsys"):            # 전에 받아 둔 것이 있으면 최신으로
        subprocess.run(["git", "-C", "/content/recsys", "pull", "-q", "--ff-only"])
    else:                                            # 처음이면 통째로 내려받는다
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/welovecherry/recsys.git", "/content/recsys"])
    os.chdir("/content/recsys/notebooks")            # 노트북 폴더 안으로 이동
    print("준비 끝 —", os.getcwd(), "· 아래 셀부터 차례로 실행하세요.")
else:
    print("내 컴퓨터에서 실행 중입니다 —", os.getcwd(), "· 따로 받아올 것이 없습니다.")

내 컴퓨터에서 실행 중입니다 — /Users/hong/workspaces/org_physical-spark/course-recsys/notebooks · 따로 받아올 것이 없습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">5 · 실습 — 먼저 추리고, 그다음 정렬한다</mark>

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.1 오늘 나오는 말  `[PPT]`</mark>

1. **2단계 추천 (two-stage)** — 추천을 **추리기**와 **정렬하기** 두 단계로 나누는 구조입니다
2. **후보 생성 (candidate generation)** — 1단계. **싼 방법**으로 후보 몇십 개만 추립니다
3. **순위 매기기 (ranking)** — 2단계. 추려진 후보**만** 모델로 정렬합니다
4. **후보 Recall (candidate recall)** — 정답이 **후보 안에 남아 있는 비율**입니다. 최종 점수의 천장이 됩니다

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">준비 · 데이터와 채점 (지난주 그대로)</mark>

**여기서 하는 일** — 데이터를 읽고, 채점에 쓸 것 세 개만 만들어 둡니다.

**채점 함수는 다시 만들지 않습니다.** `recsys.recall_at_k` 가 이미 있습니다 — 3주차에 쓰던 그 방식 그대로(시간순 분할 · 이미 담은 것은 정답에서 빼기 · 맞힐 것이 없으면 채점에서 빼기)입니다.

**채점 기준이 같아야** 오늘 숫자를 지난주 0.3918과 비교할 수 있습니다.

**쉽게 말하면 다음과 같습니다.**
1. 종목·투자자·선택 기록 세 표를 읽습니다.
2. 기록을 과거의 **학습 구간**과 마지막 한 달의 **채점 구간**으로 나눕니다.
3. 이미 담은 종목과 전체 인기순위를 미리 정리합니다.
4. 마지막에 지난주와 똑같은 채점 함수로 점수를 잽니다.

In [2]:
# 이 셀에서 하는 일 — 오늘 쓸 도구를 불러온다
# 왜 하나 — pandas 로 표를, numpy 로 숫자 묶음을 다루고, recsys.py 에는 1~4주차 함수가 있다

import sys                                  # 파이썬이 파일을 찾는 경로를 다루는 도구

sys.path.insert(0, ".")                     # 지금 폴더에서 recsys.py 를 찾게 한다
sys.path.insert(0, "notebooks")             # 한 칸 안쪽 폴더도 찾게 한다

import pandas as pd                         # 표를 다루는 도구. 앞으로 pd 라고 부른다
import numpy as np                          # 숫자 묶음을 빠르게 다루는 도구. np 라고 부른다
import recsys                               # 이 수업용으로 만든 도구 모음

print("도구 준비 완료 · pandas", pd.__version__, "· numpy", np.__version__)

도구 준비 완료 · pandas 3.0.5 · numpy 2.5.2


In [3]:
# 이 셀에서 하는 일 — 데이터를 읽고 지난주와 똑같이 학습 구간·채점 구간으로 나눈다
# 왜 하나 — 지난주와 같은 기준으로 채점해야 0.3918 과 비교할 수 있다

items, users, interactions = recsys.load()          # 종목·투자자·거래 기록 세 표
train, test, 기준시점 = recsys.split_by_time(interactions)   # 1주차부터 쓰던 시간순 분할
print(f"학습 구간 {len(train):,}건 · 채점 구간 {len(test):,}건 · 자른 날짜 {기준시점.date()}")

이름_사전 = items.set_index("item_id")["name"].to_dict()       # {종목 번호: 이름}

이미_담은것 = train.groupby("user_id")["item_id"].apply(set).to_dict()   # groupby = 사람별로 묶는다
전체_인기순위 = list(train["item_id"].value_counts().index)              # value_counts = 같은 값이 몇 번인지 센다

print(f"학습 구간에 기록이 있는 사람 {len(이미_담은것)}명 · 인기순위 {len(전체_인기순위)}개")

학습 구간 7,095건 · 채점 구간 991건 · 자른 날짜 2026-07-30
학습 구간에 기록이 있는 사람 285명 · 인기순위 100개


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.2 지난주 모델을 그대로 되살린다</mark>

**5.2 에서 하는 일** — 4주차에 만든 사람표(285×8)와 종목표(8×100)를 다시 만듭니다.

**1. 왜 다시 만드나**
- 노트북을 새로 열면 지난주 변수가 남아 있지 않습니다. 그래서 같은 코드를 한 번 더 돌립니다.
- **지난주 코드를 그대로 붙였습니다.** 새로 배울 것이 없습니다.

**2. 오늘 모델은 여기서 끝입니다**
- 이 셀 뒤로는 **모델을 한 글자도 고치지 않습니다.**
- 오늘 바꾸는 것은 이 모델로 **몇 개를 계산할지**뿐입니다.

**3. 숫자 읽는 법**
- 사람표 `(285, 8)` — 사람 285명마다 숨은 성향 점수 8개
- 종목표 `(8, 100)` — 종목 100개마다 같은 축의 점수 8개
- 지난주에는 이 둘을 통째로 곱해 **285 × 100 점수표**를 만들었습니다. 오늘은 통째로 곱하지 않습니다.

In [4]:
# 이 셀에서 하는 일 — 4주차와 똑같이 표를 만들고 요인 8개로 쪼갠다
# 왜 하나 — 오늘 쓰는 모델이 지난주 것과 같다는 것을 코드로 확인하려는 것이다

from sklearn.decomposition import TruncatedSVD      # 4주차에 쓴 행렬분해 도구

표 = pd.crosstab(train["user_id"], train["item_id"])       # crosstab = 사람 × 종목 표로 펼친다
표 = (표 > 0).astype(int)                                   # 담았으면 1, 아니면 0
표 = 표.reindex(columns=items["item_id"], fill_value=0)     # 아무도 안 담은 종목도 열로 세운다

모델 = TruncatedSVD(n_components=8, random_state=42)        # 잠재 요인 8개 — 지난주와 같은 설정
사람표 = 모델.fit_transform(표)                              # 285 × 8
종목표 = 모델.components_                                    # 8 × 100

사람_이름들 = list(표.index)                                 # 표의 세로 이름(사람) 목록
열이름 = list(표.columns)                                    # 표의 가로 이름(종목 번호) 목록

print(f"사람표 {사람표.shape} · 종목표 {종목표.shape}   ← 지난주와 같은 크기")
print(f"지난주에는 이 둘을 통째로 곱해 {사람표.shape[0]} × {종목표.shape[1]} 점수표를 만들었습니다.")

사람표 (285, 8) · 종목표 (8, 100)   ← 지난주와 같은 크기
지난주에는 이 둘을 통째로 곱해 285 × 100 점수표를 만들었습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.3 무리마다 인기 목록을 만든다 — 1단계의 재료</mark>

**5.3 에서 하는 일** — 투자 성향이 비슷한 사람들의 무리마다 **많이 담긴 순서**로 목록을 하나씩 만듭니다.

**1. 이건 2주차에 이미 해 본 것입니다**
- 2주차에 사람들을 투자 성향으로 묶었습니다. 안정형·성장형·공격형에 각각 4개씩, 무리가 12개였습니다.
- 그때는 이 목록이 **최종 추천**이었습니다. 점수가 0.2684였죠.
- 오늘은 같은 목록이 **1단계 후보**로 내려옵니다. 배운 것이 버려지지 않고 아래층으로 들어갑니다.

**2. 왜 이것이 「싼 방법」인가**
- 이 목록은 **미리 한 번만** 만들어 두면 됩니다. 12개뿐입니다.
- 사람이 접속하면 **그 사람 무리의 목록을 꺼내 오기만** 합니다. 계산이 0번입니다.
- 사람이 285명에서 100만 명으로 늘어도 **목록은 여전히 12개**입니다.

**3. 출력에서 확인할 것**
- 무리마다 종목 수가 100개가 안 됩니다(82~96개).
- **그 무리 사람이 학습 구간에 한 번도 담지 않은 종목**이 빠져 있기 때문입니다. 이것이 곧 「걸러내기」입니다.

In [5]:
# 이 셀에서 하는 일 — 무리 12개마다 많이 담긴 순서로 목록을 만든다
# 왜 하나 — 1단계에서 후보를 꺼내 쓸 재료다. 미리 만들어 두는 것이 핵심이다

무리_사전 = users.set_index("user_id")["subcluster"].to_dict()   # {사람: 무리 이름}

거래_무리 = train.copy()                                          # copy = 원본을 건드리지 않게 복사
거래_무리["무리"] = 거래_무리["user_id"].map(무리_사전)             # map = 사전을 보고 값을 바꿔 붙인다

무리_인기목록 = {}                                                # {무리 이름: 많이 담긴 종목 순서}
for 무리이름, 덩어리 in 거래_무리.groupby("무리"):                  # 무리별로 나눠 하나씩
    무리_인기목록[무리이름] = list(덩어리["item_id"].value_counts().index)   # value_counts = 같은 값이 몇 번인지 센다

print(f"무리 {len(무리_인기목록)}개의 인기 목록을 만들었습니다.\n")
for 무리이름 in sorted(무리_인기목록)[:3]:                          # 앞 세 무리만 확인
    목록 = 무리_인기목록[무리이름]                                 # 그 무리의 종목 순서를 꺼낸다
    print(f"  {무리이름:8} 종목 {len(목록):2}개 · 1위 {이름_사전[목록[0]]}")

무리 12개의 인기 목록을 만들었습니다.

  공격형-1    종목 83개 · 1위 Direxion Semiconductor Bull 3X
  공격형-2    종목 91개 · 1위 ProShares UltraPro QQQ
  공격형-3    종목 89개 · 1위 DB하이텍


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.4 후보 40개를 자른다  ✏️ 직접 해 보기</mark>

**5.4 에서 하는 일** — 무리 목록에서 **위에서 40개**를 꺼내 후보로 만듭니다.

**1. 왜 40개인가**
- 정답이 아닙니다. **재서 정한 값**입니다.
- 지난주 마지막에는 「50개」라고 예고했는데, 실제로 재 보니 40개가 더 높았습니다.
- 5.8에서 10·20·40·100을 직접 넣어 보고 왜 40인지 확인합니다.

**2. 40개가 안 되는 무리는 어떻게 하나**
- 어떤 무리는 담긴 종목이 40개가 안 될 수 있습니다.
- 그럴 때는 **전체 인기순위로 빈자리를 채웁니다.** 후보가 비어 있는 것보다는 낫습니다.

**3. 직접 채울 부분**
- 목록에서 **위에서 40개를 자르는 한 줄**입니다. `목록[:후보_개수]` 에서 대괄호 안을 채우면 됩니다.
- `[:40]` 은 **앞에서 40개까지**라는 뜻입니다.

In [6]:
# 이 셀에서 하는 일 — ✏️ 빈칸 1 · 무리 목록에서 후보 40개를 꺼낸다
# 왜 하나 — 여기가 1단계(후보 생성)다. 모델을 한 번도 쓰지 않는다는 점을 봐 두자

후보_개수 = 40                                       # 5.8 에서 이 숫자를 바꿔 본다

def 후보뽑기(사람, 이미):                               # def = 함수를 만든다. 이름을 붙여 두고 나중에 부른다
    """그 사람의 무리 목록에서 위에서 40개를 돌려준다. 모델을 쓰지 않는다."""
    내무리 = 무리_사전.get(사람)                       # get = 없으면 None 을 준다
    목록 = 무리_인기목록.get(내무리, [])                # 무리가 없는 사람은 빈 목록

    후보 = list(목록[:후보_개수])                      # ← ✏️ 빈칸 : 위에서 몇 개를 자를까

    for 번호 in 전체_인기순위:                         # 40개가 안 되면 전체 인기순으로 채운다
        if len(후보) >= 후보_개수:                     # 다 찼으면
            break                                    # 멈춘다
        if 번호 not in 후보:                           # not in = 아직 안 들어 있으면
            후보.append(번호)                          # 뒤에 붙인다
    return 후보                                       # return = 부른 쪽에 결과를 돌려준다


내_후보 = 후보뽑기("U0003", 이미_담은것["U0003"])        # 만들었으면 한 명 돌려 본다
print(f"U0003 의 후보 {len(내_후보)}개 중 앞 5개")
순위 = 1                                              # 화면에 번호를 붙이려고 세어 둔다
for 번호 in 내_후보[:5]:                                # [:5] = 앞에서 5개만
    print(f"  {순위}. {이름_사전[번호]}")
    순위 = 순위 + 1                                    # 다음 줄에 쓸 번호를 하나 올린다

U0003 의 후보 40개 중 앞 5개
  1. 포스코퓨처엠
  2. Taiwan Semiconductor
  3. KODEX 인버스
  4. Invesco QQQ Trust
  5. Palantir Technologies


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.5 후보 Recall — 오늘의 핵심 숫자</mark>

**5.5 에서 하는 일** — 방금 뽑은 후보 40개 안에 **정답이 몇 개 남아 있는지** 잽니다.

**1. 후보 Recall 이 무엇인가**
- 어떤 분이 다음 달에 종목 4개를 담았다고 합시다.
- 우리 후보 40개 안에 그 4개 중 2개만 들어 있으면 **후보 Recall 은 0.5** 입니다.

**2. 왜 이것이 「천장」인가**
- 후보에 없는 2개는 **무슨 일이 있어도 못 맞힙니다.** 2단계가 아무리 똑똑해도 목록에 없는 종목을 추천할 수는 없습니다.
- 그래서 **최종 Recall@10 은 후보 Recall 을 절대 넘지 못합니다.**
- 1단계가 놓친 것은 2단계가 못 살립니다. 오늘 가장 중요한 한 줄입니다.

**3. 어떻게 재나**
- 채점 함수를 새로 만들지 않습니다. `recsys.recall_at_k` 에 **추천 함수 대신 후보 함수**를 넣고, `k` 를 10 대신 40으로 줍니다.
- 그러면 「정답 중 후보 40개 안에 든 비율」이 그대로 나옵니다.

In [7]:
# 이 셀에서 하는 일 — 후보 40개 안에 정답이 얼마나 남았는지 잰다
# 왜 하나 — 이 값이 오늘 점수의 천장이다. 여기보다 높은 점수는 나올 수 없다

후보_recall, 인원 = recsys.recall_at_k(후보뽑기, train, test, k=후보_개수)

print(f"후보 {후보_개수}개의 후보 Recall = {후보_recall:.4f}   (채점 대상 {인원}명)")
print(f"→ 정답의 {후보_recall:.1%} 만 후보에 있습니다.")            # .1% = 소수 첫째 자리 퍼센트
print(f"→ 남은 {1 - 후보_recall:.1%} 는 오늘 무슨 일이 있어도 못 맞힙니다.")
print(f"\n오늘 Recall@10 은 {후보_recall:.4f} 을 넘을 수 없습니다. 이것이 천장입니다.")

후보 40개의 후보 Recall = 0.6460   (채점 대상 266명)
→ 정답의 64.6% 만 후보에 있습니다.
→ 남은 35.4% 는 오늘 무슨 일이 있어도 못 맞힙니다.

오늘 Recall@10 은 0.6460 을 넘을 수 없습니다. 이것이 천장입니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.6 후보에만 점수를 매긴다  ✏️ 직접 해 보기</mark>

**5.6 에서 하는 일** — 후보 40개**만** 지난주 모델로 점수 매겨 순위를 정합니다. 여기가 2단계입니다.

**1. 지난주 코드와 딱 한 군데 다릅니다**

| | 코드 | 계산하는 개수 |
|---|---|---|
| 4주차 | `사람표[내자리] @ 종목표` | 100개 |
| **5주차** | `사람표[내자리] @ 종목표[:, 후보자리]` | **40개** |

**2. `종목표[:, 후보자리]` 가 무슨 뜻인가**
- 종목표는 `8 × 100` 입니다. 가로가 종목 100개죠.
- `[:, 후보자리]` 는 **「세로는 전부, 가로는 후보 자리만」** 이라는 뜻입니다.
- 결과는 `8 × 40` 이 됩니다. 곱하면 점수가 40개만 나옵니다.

**3. 직접 채울 부분**
- 후보 종목 번호를 **표에서 몇 번째 칸인지**로 바꾸는 줄입니다.
- `열이름.index(번호)` 가 「이 종목이 표에서 몇 번째 가로 칸인가」를 알려 줍니다.

In [8]:
# 이 셀에서 하는 일 — ✏️ 빈칸 2 · 후보 40개에만 점수를 매겨 위에서 10개를 고른다
# 왜 하나 — 여기가 2단계(순위 매기기)다. 모델은 지난주 것 그대로고 개수만 줄었다

def 추천하기(사람, 이미):
    """후보 40개에만 점수를 매겨 높은 순으로 10개를 돌려준다."""
    if 사람 not in 사람_이름들:                            # 표에 줄이 없는 사람(신규)은
        return recsys.take(전체_인기순위, 이미)[:10]        # 전체 인기 목록으로 준다

    후보 = 후보뽑기(사람, 이미)                             # 1단계 — 40개로 줄인다

    후보자리 = []                                          # 후보가 표에서 몇 번째 가로 칸인지
    for 번호 in 후보:                                  # 후보를 하나씩 확인한다
        후보자리.append(열이름.index(번호))                 # ← ✏️ 빈칸 : index = 몇 번째인지 찾는다

    내자리 = 사람_이름들.index(사람)                         # 그 사람이 몇 번째 줄인지
    후보점수 = 사람표[내자리] @ 종목표[:, 후보자리]           # [:, 후보자리] = 세로 전부, 가로는 후보만

    내_순서 = []                                           # 점수가 높은 후보부터 담을 목록
    for 자리번호 in np.argsort(-후보점수):                  # argsort(-값) = 큰 것부터 자리 번호
        내_순서.append(후보[자리번호])                       # 자리 번호를 종목 번호로 바꿔 담는다
    return recsys.take(내_순서, 이미)[:10]                  # 이미 담은 것을 빼고 위에서 10개


내_추천 = 추천하기("U0003", 이미_담은것["U0003"])            # 만들었으면 한 명 돌려 본다
print(f"U0003 에게 줄 추천 10개 (후보 {후보_개수}개 중에서 고른 것)")
순위 = 1                                              # 화면에 순위를 붙이려고 세어 둔다
for 번호 in 내_추천:                                    # 추천 10개를 위에서부터
    print(f"  {순위:2}위  {이름_사전[번호]}")             # :2 = 두 칸 맞춤
    순위 = 순위 + 1                                    # 다음 줄에 쓸 순위를 하나 올린다

U0003 에게 줄 추천 10개 (후보 40개 중에서 고른 것)
   1위  TIGER 미국나스닥100레버리지
   2위  Salesforce
   3위  iShares Global Clean Energy ETF
   4위  ProShares UltraPro Short QQQ
   5위  Vanguard FTSE Developed Markets ETF
   6위  LG에너지솔루션
   7위  Global X Lithium & Battery Tech ETF
   8위  Vanguard S&P 500 ETF
   9위  TIGER 200
  10위  TIGER 2차전지테마


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.7 지난주와 같은 기준으로 채점한다</mark>

**5.7 에서 하는 일** — 방금 만든 2단계 추천을 3주차와 같은 `Recall@10` 으로 잽니다.

- 채점 코드를 다시 만들지 않고 **`recsys.recall_at_k` 한 줄**을 사용합니다.
- 재는 방법이 같으므로, 점수 차이는 **추천을 만드는 구조가 바뀐 결과**입니다.

**미리 말씀드립니다 — 점수가 오릅니다.**
- 「계산을 줄였으니 점수가 좀 떨어지겠지」 하고 보시면 놀랄 수 있습니다.
- 왜 오르는지는 다음 칸의 표에서 설명합니다.

In [9]:
# 이 셀에서 하는 일 — 3주차와 같은 기준으로 채점한다 (한 줄)
# 왜 하나 — 같은 함수를 써야 지난주 0.3918 과 공정하게 비교할 수 있다

이단계_점수, 인원 = recsys.recall_at_k(추천하기, train, test)   # 3주차 방식 그대로

print(f"5주차 2단계 추천 — Recall@10 = {이단계_점수:.4f}   (채점 대상 {인원}명)")
print(f"4주차 전부 계산   — Recall@10 = 0.3918")                # 견줄 상대
print(f"천장(후보 Recall) — {후보_recall:.4f}")                 # 넘을 수 없었던 선

print(f"\n→ {이단계_점수 - 0.3918:+.4f} ({(이단계_점수 - 0.3918) / 0.3918:+.1%}) 입니다.")
print(f"→ 계산은 100개에서 {후보_개수}개로 {1 - 후보_개수 / 100:.0%} 줄었습니다.")
print(f"→ 천장의 {이단계_점수 / 후보_recall:.0%} 를 썼습니다.")

5주차 2단계 추천 — Recall@10 = 0.4114   (채점 대상 266명)
4주차 전부 계산   — Recall@10 = 0.3918
천장(후보 Recall) — 0.6460

→ +0.0196 (+5.0%) 입니다.
→ 계산은 100개에서 40개로 60% 줄었습니다.
→ 천장의 64% 를 썼습니다.


**결과 — 덜 계산했는데 점수가 올랐습니다**

| 주차 | 방법 | 계산한 개수 | Recall@10 |
|---|---|---|---|
| 2·3주차 | 무리별 인기 목록 | — | 0.2684 |
| 4주차 | 행렬분해 · 종목 전부에 점수 | 100개 | 0.3918 |
| **5주차** | **2단계** · 무리로 추리고 모델로 정렬 | **40개** | **0.4114** |

**왜 올랐나 — 1단계가 새 정보를 넣었기 때문입니다.**

- 지난주 모델은 **거래 기록만** 봤습니다. 누가 무엇을 담았는지요.
- 그런데 무리는 **투자 성향**으로 나눈 것이었습니다. 모델이 모르던 정보입니다.
- 그래서 후보 40개가 그냥 아무 40개가 아니라 **이 사람 성향에 맞는 40개**가 되었습니다.
- 걸러내는 일이 방해가 아니라 **도움을 준 것**입니다.

> 이것이 늘 이렇게 되지는 않습니다. 후보를 너무 좁히면 무너집니다. 다음 칸에서 직접 확인합니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.8 후보 개수를 바꿔 본다</mark>

**5.8 에서 하는 일** — 후보를 10·20·40·100개로 바꿔 네 번 돌려 봅니다.

**1. 무엇을 보게 되나**
- 좁히면 **천장이 낮아서** 점수가 낮습니다.
- 넓히면 천장은 높아지는데 **점수가 다시 떨어집니다.**
- 가장 높은 지점이 **가운데**에 있습니다.

**2. 마지막 줄을 꼭 확인하세요 — 코드 검증 장치**
- 후보를 **100개**로 잡으면 종목 전체와 같습니다. 즉 **1단계가 아무것도 걸러내지 않습니다.**
- 그러면 남는 것은 2단계뿐이고, 2단계는 지난주 모델 그대로입니다.
- 따라서 **0.3918 이 나와야 맞습니다.** 지난주 점수와 소수점 네 자리까지 같아야 합니다.
- **다른 숫자가 나오면 5.4나 5.6을 잘못 채운 것입니다.**

**3. 후보 개수는 이렇게 정합니다**
- 이론으로 고르는 것이 아니라 **재서 고릅니다.** 데이터가 바뀌면 답도 바뀝니다.

In [10]:
# 이 셀에서 하는 일 — 후보 개수를 바꿔 가며 천장과 최종 점수를 함께 잰다
# 왜 하나 — 「많이 보면 좋다」가 틀렸다는 것을 직접 확인하려는 것이다

결과 = []                                              # 한 줄씩 담아 표로 만든다

for 개수 in [10, 20, 40, 100]:                        # 후보 개수를 네 가지로 바꿔 본다
    후보_개수 = 개수                                    # 위에서 만든 두 함수가 이 값을 보고 움직인다
    천장, _ = recsys.recall_at_k(후보뽑기, train, test, k=개수)   # 후보 Recall
    점수, _ = recsys.recall_at_k(추천하기, train, test)           # 최종 Recall@10
    결과.append({"후보 개수": 개수,                     # append = 목록 뒤에 한 줄 붙인다
                 "후보 Recall(천장)": round(천장, 4),
                 "Recall@10": round(점수, 4),
                 "천장을 얼마나 썼나": f"{점수 / 천장:.0%}"})

후보_개수 = 40                                          # 실험이 끝났으니 원래 값으로 되돌린다
print(pd.DataFrame(결과).to_string(index=False))        # to_string(index=False) = 왼쪽 번호 없이

 후보 개수  후보 Recall(천장)  Recall@10 천장을 얼마나 썼나
    10         0.1601     0.1437        90%
    20         0.3428     0.3106        91%
    40         0.6460     0.4114        64%
   100         1.0000     0.3918        39%

**출력 읽는 법**

1. **후보 10개** — 천장이 0.1601 뿐입니다. 정답이 후보에 거의 없어서 2단계가 할 수 있는 일이 없습니다.
   - 그런데 「천장을 얼마나 썼나」가 **90%** 입니다. 2단계는 할 일을 거의 다 했습니다.
   - **점수가 낮은 이유는 순위를 잘못 매겼기 때문이 아니라 천장이 낮았기 때문입니다.**
2. **후보 20개 → 40개** — 천장이 올라가면서 점수도 함께 오릅니다. 40개에서 가장 높습니다.
3. **후보 100개** — 천장은 1.0000 이 되지만 점수는 **0.3918** 로 내려갑니다.
   - 지난주 점수와 같습니다. 1단계가 아무것도 걸러내지 않았으니 당연합니다.
   - **여기가 0.3918 이면 오늘 코드를 제대로 짠 것입니다.**

**한 줄로 정리하면** — 좁히면 정답이 후보에 없고, 넓히면 1단계가 일을 안 합니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.9 오늘의 마무리  `[PPT]`</mark>

1. 무리 12개의 인기 목록을 미리 만들어 **1단계 후보 생성**을 했습니다. 계산은 0번이었습니다.
2. 후보 40개 안에 정답이 얼마나 남았는지 재서 **후보 Recall = 0.6460** 이라는 천장을 확인했습니다.
3. 그 40개에만 지난주 모델을 돌려 **Recall@10 = 0.4114** 를 얻었습니다. 계산은 60% 줄었습니다.
4. 후보 개수를 바꿔 보며 **가운데가 가장 높다**는 것과, 100개로 잡으면 지난주로 돌아간다는 것을 확인했습니다.

**오늘 남는 한 줄** — **1단계가 놓친 것은 2단계가 못 살립니다.**

다음 주에는 지금까지 한 번도 쓰지 않은 **종목의 섹터·자산군·위험도·테마**를 모델에 섞습니다. 좋아질지는 재 봐야 압니다.

In [11]:
# 이 셀에서 하는 일 — 오늘 점수를 점수판에 남긴다
# 왜 하나 — 5주차는 계산을 줄인 주다. 다음 주부터는 이 값과 견준다

recsys.record(5, "2단계 추천 · 후보 40개 + 순위 매기기", 이단계_점수,
              note="무리 인기순으로 후보 40개를 추리고 그 40개만 4주차 모델로 정렬했다. 계산이 100개에서 40개로 줄었다")

print("점수판에 5주차를 기록했습니다.\n")   # \n = 한 줄 띄우기
recsys.leaderboard(upto=5)   # 오늘까지 쌓인 점수판 (뒤 주차는 빼고 본다)

레벨 5 · 2단계 추천 · 후보 40개 + 순위 매기기 · Recall@10 = 0.4114
점수판에 5주차를 기록했습니다.



,level,name,recall_at_10,note
0,1,모두에게 같은 인기 순위,0.2163,알고리즘 없음. 인기 상위 10개를 모두에게 같게 추천했다
1,2,취향이 비슷한 세그먼트끼리,0.2684,거래 기록으로 주력 섹터와 평균 위험도를 뽑아 세그먼트로 나눴다
2,3,세그먼트별 목록 — 2주차와 같음,0.2684,추천 방식은 2주차와 같다. 무작위로 나누면 0.3150 이 나오는데 그것은 미래를...
3,4,행렬분해 SVD · 요인 8개,0.3918,규칙을 사람이 정하지 않았다. 사람 285 × 종목 100 표를 요인 8개로 쪼갰다
4,5,2단계 추천 · 후보 40개 + 순위 매기기,0.4114,무리 인기순으로 후보 40개를 추리고 그 40개만 4주차 모델로 정렬했다. 계산이 ...


## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">파이썬 문법 · 대괄호로 잘라 쓰기 (슬라이싱)</mark>

오늘 빈칸에 쓴 `목록[:40]` 과 `종목표[:, 후보자리]` 를 조금 더 봅니다.

1. **목록 자르기** — `목록[:40]` 은 **앞에서 40개까지**입니다.
   - `[:3]` 앞에서 3개 · `[3:]` 4번째부터 끝까지 · `[-1]` 맨 마지막 하나
2. **표 자르기** — 표는 세로·가로 두 방향이라 쉼표로 나눠 씁니다.
   - `표[:, 3]` 은 **세로는 전부, 가로는 4번째 칸**입니다. 콜론 하나만 쓰면 「전부」라는 뜻입니다.
   - 오늘 쓴 `종목표[:, 후보자리]` 는 **세로 전부, 가로는 후보 자리들만** 꺼낸 것입니다.

아래 셀은 **마음껏 고쳐 보세요.** 위 실습에 영향을 주지 않습니다.

In [12]:
# 파이썬 문법 연습 · 마음껏 고쳐 보세요. 이 셀은 위 실습에 영향을 주지 않습니다.

연습_목록 = ["가", "나", "다", "라", "마"]
print("앞에서 3개    :", 연습_목록[:3])      # [:3] = 앞에서 3개까지
print("4번째부터 끝  :", 연습_목록[3:])      # [3:] = 4번째부터 끝까지
print("맨 마지막     :", 연습_목록[-1])      # [-1] = 맨 마지막 하나

연습_표 = np.array([[1, 2, 3, 4],            # 세로 2줄 × 가로 4칸
                    [5, 6, 7, 8]])          # 대괄호 안의 대괄호 하나가 한 줄이다
print("\n표 전체      :\n", 연습_표)
print("세로 전부, 가로 3번째 칸 :", 연습_표[:, 2])         # 콜론 하나 = 그 방향은 전부
print("세로 전부, 가로 1·4번째 :\n", 연습_표[:, [0, 3]])   # 자리 목록을 주면 그 칸들만

앞에서 3개    : ['가', '나', '다']
4번째부터 끝  : ['라', '마']
맨 마지막     : 마

표 전체      :
 [[1 2 3 4]
 [5 6 7 8]]
세로 전부, 가로 3번째 칸 : [3 7]
세로 전부, 가로 1·4번째 :
 [[1 4]
 [5 8]]
